In [1]:
import math

In [2]:
from dataclasses import dataclass
import numpy as np

@dataclass
class CombinedResult:
    R_comb: float                 # combined central value
    weights: np.ndarray           # BLUE weights (sum to 1)
    sigma_stat: float             # absolute statistical uncertainty
    sigma_sys_unc: float          # absolute uncorrelated systematic
    sigma_sys_com: float          # absolute fully-correlated (multiplicative) systematic
    sigma_sys_tot: float          # sqrt(sys_unc^2 + sys_com^2)
    sigma_tot: float              # sqrt(stat^2 + sys_tot^2)

def _build_common_cov(R: np.ndarray, c_common):
    """
    Build the covariance matrix for fully-correlated multiplicative systematics.
    - If c_common is a scalar: uses same relative error for all points.
    - If c_common is a 1D array of length n: per-point relative errors (still fully correlated, rho=1).
    - If c_common is a 2D array/list with shape (k, n): k fully-correlated sources; sum their covariances.
    Returns an (n x n) covariance matrix C.
    """
    R = np.asarray(R, float).reshape(-1)
    n = R.size

    if c_common is None:
        return np.zeros((n, n), dtype=float)

    c_arr = np.asarray(c_common, dtype=float)

    # Case 1: scalar
    if c_arr.ndim == 0:
        v = R * float(c_arr)               # absolute fully-correlated error per point
        return np.outer(v, v)

    # Case 2: 1D vector of length n
    if c_arr.ndim == 1:
        if c_arr.size != n:
            raise ValueError("c_common vector must have same length as R")
        v = R * c_arr
        return np.outer(v, v)

    # Case 3: 2D array: rows are different fully-correlated sources
    if c_arr.ndim == 2:
        if c_arr.shape[1] != n:
            raise ValueError("c_common 2D must have shape (k, n)")
        C = np.zeros((n, n), dtype=float)
        for row in c_arr:
            v = R * row
            C += np.outer(v, v)
        return C

    raise ValueError("Unsupported shape for c_common")

def combine_ratio_with_correlated_norm(R, s_stat, u_unc, c_common=None, use_C_in_weights=False):
    """
    Combine multiple measurements of R with:
      - uncorrelated statistical relative errors s_stat (array-like)
      - uncorrelated systematic relative errors u_unc (array-like)
      - fully-correlated *multiplicative* relative error(s) c_common:
          * scalar: same for all points
          * 1D array (len n): per-point relative but fully correlated (rho=1)
          * 2D array (k x n): k fully-correlated sources; summed

    Returns a CombinedResult with full stat/syst breakdown.
    """
    R = np.asarray(R, dtype=float).reshape(-1)
    s = np.asarray(s_stat, dtype=float).reshape(-1)
    u = np.asarray(u_unc, dtype=float).reshape(-1)
    n = R.size
    if not (s.size == n and u.size == n):
        raise ValueError("R, s_stat, u_unc must have the same length")

    # Absolute uncorrelated pieces
    sig_stat = s * R
    sig_unc  = u * R
    S = np.diag(sig_stat**2)
    U = np.diag(sig_unc**2)

    # Fully-correlated multiplicative piece(s)
    C = _build_common_cov(R, c_common)

    # Full covariance
    # V = S + U + C
    V = S + U if not use_C_in_weights else (S + U + C)

    # BLUE weights
    one = np.ones(n)
    Vinv = np.linalg.inv(V)
    w = Vinv @ one / (one @ Vinv @ one)

    # Combined central value
    R_comb = float(w @ R)

    # Error decomposition with same weights
    var_stat = float(w @ S @ w)
    var_unc  = float(w @ U @ w)
    var_com  = float(w @ C @ w)   # if c_common is a vector: equals (sum_i w_i R_i c_i)^2

    sigma_stat = np.sqrt(var_stat)
    sigma_sys_unc = np.sqrt(var_unc)
    sigma_sys_com = np.sqrt(var_com)
    sigma_sys_tot = np.sqrt(var_unc + var_com)
    sigma_tot = np.sqrt(var_stat + var_unc + var_com)

    return CombinedResult(
        R_comb=R_comb,
        weights=w,
        sigma_stat=sigma_stat,
        sigma_sys_unc=sigma_sys_unc,
        sigma_sys_com=sigma_sys_com,
        sigma_sys_tot=sigma_sys_tot,
        sigma_tot=sigma_tot,
    )




In [3]:
def decompose_common_sources(R, weights, c_common, source_labels=None):
    """
    Decompose sigma_sys_com into contributions from each fully-correlated source.

    R            : array of central values (length n)
    weights      : BLUE weights used for the final combination (length n)
    c_common     : (k x n) array (or list of lists), each row = per-point *relative* error of source k
                   (still fully correlated, rho=1 for every source)
    source_labels: optional list of k labels (strings)

    Returns:
      sigmas_k      : array (k,) of absolute sigma from each source (std dev, not variance)
      sigma_com_tot : scalar, sqrt(sum_k sigmas_k^2), should match resB.sigma_sys_com
      fracs_k       : array (k,) of variance fractions (sigmas_k^2 / sum sigmas_k^2)
    """
    R = np.asarray(R, float).reshape(-1)
    w = np.asarray(weights, float).reshape(-1)
    C = np.asarray(c_common, float)

    if C.ndim == 1:
        C = C[None, :]  # make it (1, n)

    if C.shape[1] != R.size:
        raise ValueError("c_common must have shape (k, n) where n == len(R)")

    # For a fully-correlated multiplicative source k:
    # variance contribution = ( sum_i w_i * R_i * c_{k,i} )^2
    # -> std dev contribution = abs( sum_i w_i * R_i * c_{k,i} )
    t = w * R                      # length-n helper
    sigmas_k = np.abs(C @ t)       # length-k
    var_k = sigmas_k**2
    sigma_com_tot = float(np.sqrt(np.sum(var_k)))
    fracs_k = var_k / np.sum(var_k)

    # Pretty print
    if source_labels is None:
        source_labels = [f"source #{i+1}" for i in range(C.shape[0])]
    print("== Fully-correlated (multiplicative) source breakdown ==")
    for lbl, sig_k, frac in zip(source_labels, sigmas_k, fracs_k):
        print(f"  {lbl:>12s}: sigma = {sig_k:.12e}   (variance share {frac*100:6.2f}%)")
    print(f"  --> Quadrature sum (should match sigma_sys_com): {sigma_com_tot:.12e}")
    return sigmas_k, sigma_com_tot, fracs_k

In [4]:
# ---------------------------
# Br: D+ -> eta K+
# ---------------------------
# Two measurements 
R  = [1.0730e-04, 1.0984e-04]
s  = [1.1098e-05/R[0], 1.1080e-05/R[1]]   # stat (relative, uncorrelated)
u  = [math.sqrt(0.729**2 + 1.958**2 + 1.866**2 + 0.168**2 + 0.142**2 + 0.042**2)/100, math.sqrt(0.450**2 + 1.170**2 + 1.048**2 + 0.167**2 + 0.143**2 + 0.351**2)/100]    # syst (relative, uncorrelated)

# (B) multiple fully-correlated sources at once (each a per-point relative vector)
# e.g., luminosity [%], common calibration [%]
c_sources = np.array([
    [0.417/100, 0.433/100],  # source #1, same 1% on both
    # [2.387/100, 2.387/100],  # source #2, 1.5% vs 2.0%, still fully correlated
    [2.4/100, 2.4/100],  # source #2, 1.5% vs 2.0%, still fully correlated
])
resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources, use_C_in_weights=True)
print("Two fully-correlated sources summed:")
print("  weights =", resB.weights)
print("  R_comb  =", f"{resB.R_comb:.12e}")
print("  sigma_stat   =", f"{resB.sigma_stat:.12e}")
print("  sigma_sys_unc=", f"{resB.sigma_sys_unc:.12e}")
print("  sigma_sys_com=", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_tot=", f"{resB.sigma_sys_tot:.12e}")
print("  sigma_stat_sys_tot=", f"{resB.sigma_tot:.12e}")


print("\n")

# Using your existing definitions:
# R, s, u defined above; resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources)

labels = ["Frist common", "Second common"]
sig_k, sigma_check, fracs = decompose_common_sources(R, resB.weights, c_sources, source_labels=labels)

# Optional sanity check against the aggregator from resB
print("\nCross-check:")
print("  sigma_sys_com (from resB) =", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_com (decomposed sum) =", f"{sigma_check:.12e}")

Two fully-correlated sources summed:
  weights = [0.48893429 0.51106571]
  R_comb  = 1.085981069007e-04
  sigma_stat   = 7.842748107489e-06
  sigma_sys_unc= 1.751836281323e-06
  sigma_sys_com= 2.646956064699e-06
  sigma_sys_tot= 3.174162372187e-06
  sigma_stat_sys_tot= 8.460733103138e-06


== Fully-correlated (multiplicative) source breakdown ==
  Frist common: sigma = 4.618357789710e-07   (variance share   3.04%)
  Second common: sigma = 2.606354565617e-06   (variance share  96.96%)
  --> Quadrature sum (should match sigma_sys_com): 2.646956064699e-06

Cross-check:
  sigma_sys_com (from resB) = 2.646956064699e-06
  sigma_sys_com (decomposed sum) = 2.646956064699e-06


In [5]:
print("Final results:")
print(f"R = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {resB.sigma_sys_tot:.12e} (total sys)")
print(f"\nR = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {math.sqrt(resB.sigma_sys_tot**2 - sig_k[1]**2):.12e} (sys without norm Br) pm {sig_k[1]:.12e} (sys norm Br)")

# total_R1_sys_unc = math.sqrt( (u[0]*R[0]) **2 + (c_sources[0][0]*R[0])**2 +  (c_sources[1][0]*R[0])**2 )
# total_R2_sys_unc = math.sqrt( (u[1]*R[1]) **2 + (c_sources[0][1]*R[1])**2 +  (c_sources[1][1]*R[1])**2 )



R1_sys_without_norm_Br = math.sqrt( (u[0]*R[0]) **2 + (c_sources[0][0]*R[0])**2)
R2_sys_without_norm_Br = math.sqrt( (u[1]*R[1]) **2 + (c_sources[0][1]*R[1])**2)

R1_sys_norm_Br = c_sources[1][0]*R[0]
R2_sys_norm_Br = c_sources[1][1]*R[1]

total_R1_sys_unc = math.sqrt( R1_sys_without_norm_Br**2 + R1_sys_norm_Br**2 )
total_R2_sys_unc = math.sqrt( R2_sys_without_norm_Br**2 + R2_sys_norm_Br**2 )

print(f"\nR1 = {R[0]:.12e} pm {s[0]*R[0]:.12e} (stat) pm {total_R1_sys_unc:.12e} (total sys)")
print(f"R2 = {R[1]:.12e} pm {s[1]*R[1]:.12e} (stat) pm {total_R2_sys_unc:.12e} (total sys)")

print(f"\nR1 = {R[0]:.12e} pm {s[0]*R[0]:.12e} (stat) pm {R1_sys_without_norm_Br:.12e} (sys without norm Br) pm {R1_sys_norm_Br:.12e} (sys norm Br)")
print(f"R2 = {R[1]:.12e} pm {s[1]*R[1]:.12e} (stat) pm {R2_sys_without_norm_Br:.12e} (sys without norm Br) pm {R2_sys_norm_Br:.12e} (sys norm Br)")

# print(f"x1 = {R[0]:.8e} pm (stat) pm (sys)")

Final results:
R = 1.085981069007e-04 pm 7.842748107489e-06 (stat) pm 3.174162372187e-06 (total sys)

R = 1.085981069007e-04 pm 7.842748107489e-06 (stat) pm 1.811690548438e-06 (sys without norm Br) pm 2.606354565617e-06 (sys norm Br)

R1 = 1.073000000000e-04 pm 1.109800000000e-05 (stat) pm 3.990521766318e-06 (total sys)
R2 = 1.098400000000e-04 pm 1.108000000000e-05 (stat) pm 3.256294194712e-06 (total sys)

R1 = 1.073000000000e-04 pm 1.109800000000e-05 (stat) pm 3.048378081449e-06 (sys without norm Br) pm 2.575200000000e-06 (sys norm Br)
R2 = 1.098400000000e-04 pm 1.108000000000e-05 (stat) pm 1.911573262241e-06 (sys without norm Br) pm 2.636160000000e-06 (sys norm Br)


In [6]:
# ---------------------------
# Two measurements 
R  = [2.8614e-02, 2.9290e-02]
s  = [2.9595e-03/R[0], 2.9546e-03/R[1]]   # stat (relative, uncorrelated)
u  = [math.sqrt(0.729**2 + 1.958**2 + 1.866**2 + 0.168**2 + 0.142**2 + 0.042**2)/100, math.sqrt(0.450**2 + 1.170**2 + 1.048**2 + 0.167**2 + 0.143**2 + 0.351**2)/100]    # syst (relative, uncorrelated)

# (B) multiple fully-correlated sources at once (each a per-point relative vector)
# e.g., luminosity [%], common calibration [%]
c_sources = np.array([
    [0.417/100, 0.433/100],  # source #1, same 1% on both
    # [2.387/100, 2.387/100],  # source #2, 1.5% vs 2.0%, still fully correlated
])
resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources, use_C_in_weights=True)
print("Two fully-correlated sources summed:")
print("  weights =", resB.weights)
print("  R_comb  =", f"{resB.R_comb:.12e}")
print("  sigma_stat   =", f"{resB.sigma_stat:.12e}")
print("  sigma_sys_unc=", f"{resB.sigma_sys_unc:.12e}")
print("  sigma_sys_com=", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_tot=", f"{resB.sigma_sys_tot:.12e}")
print("  sigma_stat_sys_tot=", f"{resB.sigma_tot:.12e}")


print("\n")

# Using your existing definitions:
# R, s, u defined above; resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources)

labels = ["First common", "Second common"]
sig_k, sigma_check, fracs = decompose_common_sources(R, resB.weights, c_sources, source_labels=labels)

# Optional sanity check against the aggregator from resB
print("\nCross-check:")
print("  sigma_sys_com (from resB) =", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_com (decomposed sum) =", f"{sigma_check:.12e}")

Two fully-correlated sources summed:
  weights = [0.48830219 0.51169781]
  R_comb  = 2.895990772224e-02
  sigma_stat   = 2.091441952237e-03
  sigma_sys_unc= 4.669020453013e-04
  sigma_sys_com= 1.231608358371e-04
  sigma_sys_tot= 4.828727693613e-04
  sigma_stat_sys_tot= 2.146461169220e-03


== Fully-correlated (multiplicative) source breakdown ==
  First common: sigma = 1.231608358371e-04   (variance share 100.00%)
  --> Quadrature sum (should match sigma_sys_com): 1.231608358371e-04

Cross-check:
  sigma_sys_com (from resB) = 1.231608358371e-04
  sigma_sys_com (decomposed sum) = 1.231608358371e-04


In [7]:
print("Final results:")
print(f"R = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {resB.sigma_sys_tot:.12e} (total sys)")
# print(f"\nR = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {math.sqrt(resB.sigma_sys_tot**2 - sig_k[1]**2):.12e} (sys without norm Br) pm {sig_k[1]:.12e} (sys norm Br)")


R1_sys_without_norm_Br = math.sqrt( (u[0]*R[0]) **2 + (c_sources[0][0]*R[0])**2)
R2_sys_without_norm_Br = math.sqrt( (u[1]*R[1]) **2 + (c_sources[0][1]*R[1])**2)


total_R1_sys_unc = math.sqrt( R1_sys_without_norm_Br**2  )
total_R2_sys_unc = math.sqrt( R2_sys_without_norm_Br**2 )

print(f"\nR1 = {R[0]:.12e} pm {s[0]*R[0]:.12e} (stat) pm {total_R1_sys_unc:.12e} (total sys)")
print(f"R2 = {R[1]:.12e} pm {s[1]*R[1]:.12e} (stat) pm {total_R2_sys_unc:.12e} (total sys)")

Final results:
R = 2.895990772224e-02 pm 2.091441952237e-03 (stat) pm 4.828727693613e-04 (total sys)

R1 = 2.861400000000e-02 pm 2.959500000000e-03 (stat) pm 8.129197616270e-04 (total sys)
R2 = 2.929000000000e-02 pm 2.954600000000e-03 (stat) pm 5.097412677625e-04 (total sys)


In [8]:
# ---------------------------
# Br: Ds+ -> eta K+
# ---------------------------
# Two measurements 
R  = [1.5946e-03, 1.4157e-03]
s  = [4.5181e-05/R[0], 4.6232e-05/R[1]]   # stat (relative, uncorrelated)
u  = [math.sqrt(0.729**2 + 1.474**2 + 1.473**2 + 0.180**2 + 0.150**2 + 0.083**2)/100, math.sqrt(0.450**2 + 1.606**2 + 1.370**2 + 0.179**2 + 0.152**2 + 0.061**2)/100]    # syst (relative, uncorrelated)

# (B) multiple fully-correlated sources at once (each a per-point relative vector)
# e.g., luminosity [%], common calibration [%]
c_sources = np.array([
    [0.406/100, 0.411/100],  # source #1, same 1% on both
    [1.601/100, 1.601/100],  # source #2, 1.5% vs 2.0%, still fully correlated
])
resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources, use_C_in_weights=True)
print("Two fully-correlated sources summed:")
print("  weights =", resB.weights)
print("  R_comb  =", f"{resB.R_comb:.12e}")
print("  sigma_stat   =", f"{resB.sigma_stat:.12e}")
print("  sigma_sys_unc=", f"{resB.sigma_sys_unc:.12e}")
print("  sigma_sys_com=", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_tot=", f"{resB.sigma_sys_tot:.12e}")
print("  sigma_stat_sys_tot=", f"{resB.sigma_tot:.12e}")


print("\n")

# Using your existing definitions:
# R, s, u defined above; resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources)

labels = ["First common", "Second common"]
sig_k, sigma_check, fracs = decompose_common_sources(R, resB.weights, c_sources, source_labels=labels)

# Optional sanity check against the aggregator from resB
print("\nCross-check:")
print("  sigma_sys_com (from resB) =", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_com (decomposed sum) =", f"{sigma_check:.12e}")




Two fully-correlated sources summed:
  weights = [0.47185703 0.52814297]
  R_comb  = 1.500115222008e-03
  sigma_stat   = 3.241440487829e-05
  sigma_sys_unc= 2.330569005253e-05
  sigma_sys_com= 2.478627452059e-05
  sigma_sys_tot= 3.402226614197e-05
  sigma_stat_sys_tot= 4.699157623499e-05


== Fully-correlated (multiplicative) source breakdown ==
  First common: sigma = 6.127852401744e-06   (variance share   6.11%)
  Second common: sigma = 2.401684470435e-05   (variance share  93.89%)
  --> Quadrature sum (should match sigma_sys_com): 2.478627452059e-05

Cross-check:
  sigma_sys_com (from resB) = 2.478627452059e-05
  sigma_sys_com (decomposed sum) = 2.478627452059e-05


In [9]:
print("Final results:")
print(f"R = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {resB.sigma_sys_tot:.12e} (total sys)")
print(f"\nR = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {math.sqrt(resB.sigma_sys_tot**2 - sig_k[1]**2):.12e} (sys without norm Br) pm {sig_k[1]:.12e} (sys norm Br)")

# total_R1_sys_unc = math.sqrt( (u[0]*R[0]) **2 + (c_sources[0][0]*R[0])**2 +  (c_sources[1][0]*R[0])**2 )
# total_R2_sys_unc = math.sqrt( (u[1]*R[1]) **2 + (c_sources[0][1]*R[1])**2 +  (c_sources[1][1]*R[1])**2 )



R1_sys_without_norm_Br = math.sqrt( (u[0]*R[0]) **2 + (c_sources[0][0]*R[0])**2)
R2_sys_without_norm_Br = math.sqrt( (u[1]*R[1]) **2 + (c_sources[0][1]*R[1])**2)

R1_sys_norm_Br = c_sources[1][0]*R[0]
R2_sys_norm_Br = c_sources[1][1]*R[1]

total_R1_sys_unc = math.sqrt( R1_sys_without_norm_Br**2 + R1_sys_norm_Br**2 )
total_R2_sys_unc = math.sqrt( R2_sys_without_norm_Br**2 + R2_sys_norm_Br**2 )

print(f"\nR1 = {R[0]:.12e} pm {s[0]*R[0]:.12e} (stat) pm {total_R1_sys_unc:.12e} (total sys)")
print(f"R2 = {R[1]:.12e} pm {s[1]*R[1]:.12e} (stat) pm {total_R2_sys_unc:.12e} (total sys)")

print(f"\nR1 = {R[0]:.12e} pm {s[0]*R[0]:.12e} (stat) pm {R1_sys_without_norm_Br:.12e} (sys without norm Br) pm {R1_sys_norm_Br:.12e} (sys norm Br)")
print(f"R2 = {R[1]:.12e} pm {s[1]*R[1]:.12e} (stat) pm {R2_sys_without_norm_Br:.12e} (sys without norm Br) pm {R2_sys_norm_Br:.12e} (sys norm Br)")


Final results:
R = 1.500115222008e-03 pm 3.241440487829e-05 (stat) pm 3.402226614197e-05 (total sys)

R = 1.500115222008e-03 pm 3.241440487829e-05 (stat) pm 2.409783732791e-05 (sys without norm Br) pm 2.401684470435e-05 (sys norm Br)

R1 = 1.594600000000e-03 pm 4.518100000000e-05 (stat) pm 4.414385069352e-05 (total sys)
R2 = 1.415700000000e-03 pm 4.623200000000e-05 (stat) pm 3.864011486637e-05 (total sys)

R1 = 1.594600000000e-03 pm 4.518100000000e-05 (stat) pm 3.601280098917e-05 (sys without norm Br) pm 2.552954600000e-05 (sys norm Br)
R2 = 1.415700000000e-03 pm 4.623200000000e-05 (stat) pm 3.129440954785e-05 (sys without norm Br) pm 2.266535700000e-05 (sys norm Br)


In [10]:
# ---------------------------
# Br: Ds+ -> eta K+
# ---------------------------
# Two measurements 
R  = [9.4577e-02,  8.3968e-02]
s  = [2.6798e-03/R[0], 2.7421e-03/R[1]]   # stat (relative, uncorrelated)
u  = [math.sqrt(0.729**2 + 1.474**2 + 1.473**2 + 0.180**2 + 0.150**2 + 0.083**2)/100, math.sqrt(0.450**2 + 1.606**2 + 1.370**2 + 0.179**2 + 0.152**2 + 0.061**2)/100]    # syst (relative, uncorrelated)

# (B) multiple fully-correlated sources at once (each a per-point relative vector)
# e.g., luminosity [%], common calibration [%]
c_sources = np.array([
    [0.406/100, 0.411/100],  # source #1, same 1% on both
    # [1.601/100, 1.601/100],  # source #2, 1.5% vs 2.0%, still fully correlated
])
resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources, use_C_in_weights=True)
print("Two fully-correlated sources summed:")
print("  weights =", resB.weights)
print("  R_comb  =", f"{resB.R_comb:.12e}")
print("  sigma_stat   =", f"{resB.sigma_stat:.12e}")
print("  sigma_sys_unc=", f"{resB.sigma_sys_unc:.12e}")
print("  sigma_sys_com=", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_tot=", f"{resB.sigma_sys_tot:.12e}")
print("  sigma_stat_sys_tot=", f"{resB.sigma_tot:.12e}")


print("\n")

# Using your existing definitions:
# R, s, u defined above; resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources)

labels = ["First common", "(1.60%, 1.60%) common"]
sig_k, sigma_check, fracs = decompose_common_sources(R, resB.weights, c_sources, source_labels=labels)

# Optional sanity check against the aggregator from resB
print("\nCross-check:")
print("  sigma_sys_com (from resB) =", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_com (decomposed sum) =", f"{sigma_check:.12e}")

Two fully-correlated sources summed:
  weights = [0.48263916 0.51736084]
  R_comb  = 8.908831882785e-02
  sigma_stat   = 1.919740875523e-03
  sigma_sys_unc= 1.385161960005e-03
  sigma_sys_com= 3.638706621999e-04
  sigma_sys_tot= 1.432157642948e-03
  sigma_stat_sys_tot= 2.395095101119e-03


== Fully-correlated (multiplicative) source breakdown ==
  First common: sigma = 3.638706621999e-04   (variance share 100.00%)
  --> Quadrature sum (should match sigma_sys_com): 3.638706621999e-04

Cross-check:
  sigma_sys_com (from resB) = 3.638706621999e-04
  sigma_sys_com (decomposed sum) = 3.638706621999e-04


In [11]:
print("Final results:")
print(f"R = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {resB.sigma_sys_tot:.12e} (total sys)")
# print(f"\nR = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {math.sqrt(resB.sigma_sys_tot**2 - sig_k[1]**2):.12e} (sys without norm Br) pm {sig_k[1]:.12e} (sys norm Br)")


R1_sys_without_norm_Br = math.sqrt( (u[0]*R[0]) **2 + (c_sources[0][0]*R[0])**2)
R2_sys_without_norm_Br = math.sqrt( (u[1]*R[1]) **2 + (c_sources[0][1]*R[1])**2)


total_R1_sys_unc = math.sqrt( R1_sys_without_norm_Br**2  )
total_R2_sys_unc = math.sqrt( R2_sys_without_norm_Br**2 )

print(f"\nR1 = {R[0]:.12e} pm {s[0]*R[0]:.12e} (stat) pm {total_R1_sys_unc:.12e} (total sys)")
print(f"R2 = {R[1]:.12e} pm {s[1]*R[1]:.12e} (stat) pm {total_R2_sys_unc:.12e} (total sys)")


Final results:
R = 8.908831882785e-02 pm 1.919740875523e-03 (stat) pm 1.432157642948e-03 (total sys)

R1 = 9.457700000000e-02 pm 2.679800000000e-03 (stat) pm 2.135947998967e-03 (total sys)
R2 = 8.396800000000e-02 pm 2.742100000000e-03 (stat) pm 1.856134054470e-03 (total sys)


In [14]:
from dataclasses import dataclass
import numpy as np

@dataclass
class AcpCombineResult:
    A_comb: float              # combined central value
    weights: np.ndarray        # BLUE weights (sum to 1)
    sigma_stat: float          # absolute statistical uncertainty
    sigma_syst_unc: float      # absolute uncorrelated systematic
    sigma_syst_com: float      # absolute fully-correlated additive systematic
    sigma_syst_tot: float      # sqrt(syst_unc^2 + syst_com^2)
    sigma_tot: float           # sqrt(stat^2 + syst_tot^2)

def _outer_sum(v_list):
    """Sum of outer products: sum_k v_k v_k^T."""
    if not v_list:
        return None
    C = np.zeros((v_list[0].size, v_list[0].size), dtype=float)
    for v in v_list:
        C += np.outer(v, v)
    return C

def build_cov_additive_common(d_common, n):
    """
    Build K for fully-correlated *additive* systematics with possibly different magnitudes.
    - d_common: None | scalar | 1D (n,) | 2D (k,n)
      values are absolute sigmas (same unit as the measurement).
    """
    if d_common is None:
        return np.zeros((n, n), dtype=float)

    d = np.asarray(d_common, dtype=float)
    if d.ndim == 0:            # scalar -> same absolute size for all points
        v = np.full(n, float(d), dtype=float)
        return np.outer(v, v)
    if d.ndim == 1:            # per-point absolute sizes
        if d.size != n:
            raise ValueError("d_common vector must have length n")
        return np.outer(d, d)  # fully correlated -> rank-1
    if d.ndim == 2:            # multiple fully-correlated additive sources (independent)
        if d.shape[1] != n:
            raise ValueError("d_common 2D must have shape (k, n)")
        v_list = [row.astype(float) for row in d]
        return _outer_sum(v_list)
    raise ValueError("Unsupported shape for d_common")

def combine_acp_two_matrix(a, b_stat, c_unc, d_common,
                           x, y_stat, z_unc):
    """
    Combine two A_CP-like measurements using full matrix calculus.
      m1 = a ± b_stat (stat) ± c_unc (uncorr syst) ± d1 (common additive)
      m2 = x ± y_stat (stat) ± z_unc (uncorr syst) ± d2 (common additive)
    Here d_common can be:
      - scalar d  -> d1=d2=d
      - vector [d1, d2] -> different magnitudes but fully correlated (rho=1)
      - 2D (k,2) -> multiple independent additive common sources; each row is [d1_k, d2_k]
    Returns AcpCombineResult with BLUE weights and an error breakdown via quadratic forms.
    """
    # data vector
    y = np.array([a, x], dtype=float)
    n = y.size

    # diagonal pieces
    S = np.diag([b_stat**2, y_stat**2])    # stat (absolute)
    U = np.diag([c_unc**2,  z_unc**2 ])    # uncorrelated syst (absolute)

    # fully-correlated additive piece(s)
    K = build_cov_additive_common(d_common, n)

    # full covariance and GLS weights
    V = S + U + K
    one = np.ones(n)
    Vinv = np.linalg.inv(V)
    w = Vinv @ one / (one @ Vinv @ one)

    # combined central value
    A_comb = float(w @ y)

    # error decomposition (all via matrices)
    var_stat = float(w @ S @ w)
    var_unc  = float(w @ U @ w)
    var_com  = float(w @ K @ w)

    sigma_stat    = np.sqrt(var_stat)
    sigma_syst_unc= np.sqrt(var_unc)
    sigma_syst_com= np.sqrt(var_com)
    sigma_syst_tot= np.sqrt(var_unc + var_com)
    sigma_tot     = np.sqrt(var_stat + var_unc + var_com)

    return AcpCombineResult(
        A_comb=A_comb,
        weights=w,
        sigma_stat=sigma_stat,
        sigma_syst_unc=sigma_syst_unc,
        sigma_syst_com=sigma_syst_com,
        sigma_syst_tot=sigma_syst_tot,
        sigma_tot=sigma_tot,
    )

# (Optional) Jacobian-based propagation check: M Vx M^T = V
def propagate_with_jacobian(b_stat, y_stat, c_unc, z_unc, d_common):
    """
    Build V via M Vx M^T with additive fully-correlated offsets:
      y = x + a * z,  z~N(0, tau^2=1),  a = [d1, d2]
    Returns V (should equal S+U+K).
    """
    S = np.diag([b_stat**2, y_stat**2])
    U = np.diag([c_unc**2,  z_unc**2 ])
    if d_common is None:
        a = np.zeros(2)
    else:
        d = np.asarray(d_common, float)
        if d.ndim == 0:
            a = np.array([float(d), float(d)], float)
        elif d.ndim == 1 and d.size == 2:
            a = d.astype(float)
        else:
            raise ValueError("Jacobian check supports scalar or 1D len-2 d_common only.")
    M  = np.column_stack([np.eye(2), a.reshape(2,1)])  # [ I | a ]
    Vx = np.zeros((3,3), float)
    Vx[:2,:2] = S + U          # diag for x1,x2
    Vx[2,2]   = 1.0            # Var(z)=tau^2
    V = M @ Vx @ M.T
    return V

def print_acp_result(res: AcpCombineResult, label="Acp"):
    """Pretty-print the combination result with a clear breakdown."""
    # helper to format relative (%) if central value is non-zero
    def rel(x):
        return (x / res.A_comb * 100.0) if res.A_comb != 0 else float("nan")

    print(f"== {label} combination ==")
    # print(f"Weights: w1 = {res.w1:.4f}, w2 = {res.w2:.4f}")
    print(f"Weights: {res.weights}")
    print(f"{label} = {res.A_comb:.6e}")
    print(f"  stat         : {res.sigma_stat:.6e}  (rel {rel(res.sigma_stat):.3f}%)")
    print(f"  syst (uncorr): {res.sigma_syst_unc:.6e}  (rel {rel(res.sigma_syst_unc):.3f}%)")
    print(f"  syst (common): {res.sigma_syst_com:.6e}  (rel {rel(res.sigma_syst_com):.3f}%)")
    print(f"  syst (total) : {res.sigma_syst_tot:.6e}  (rel {rel(res.sigma_syst_tot):.3f}%)")
    print(f"  TOTAL        : {res.sigma_tot:.6e}     (rel {rel(res.sigma_tot):.3f}%)")
    print()
    # compact “paper-style” line
    print(f"{label} = ({res.A_comb:.6e} ± {res.sigma_stat:.6e} (stat) "
          f"± {res.sigma_syst_unc:.6e} (syst-unc) ± {res.sigma_syst_com:.6e} (syst-com))")
    print(f"{label} = ({res.A_comb:.6e} ± {res.sigma_stat:.6e} (stat) "
          f"± {res.sigma_syst_tot:.6e} (syst-unc-tot)")
    print(f"{label} = ({res.A_comb:.6e} ± {res.sigma_tot:.6e})  [total]")


In [15]:
# Acp: D+ -> eta pi+
a, b, c, d =  -0.20586e-2, 0.65458e-2, math.sqrt(0.020**2 + 0.0086**2 + 0.000)*1e-2, [0.000068,0.000068]   # result1: a ± b (± c ± d)
x, y, z     =   0.15044e-2, 0.83449e-2, math.sqrt(0.022**2 + 0.0092**2 + 0.0102**2)*1e-2         # result2: x ± y (± z ± d)

res = combine_acp_two_matrix(a, b, c, d, x, y, z)
print(f"eta->gg")
print(f"{a:.6e} ± {b:.6e} ± {math.sqrt(c**2 + d[0]**2):.6e}")
print(f"eta->pipipi")
print(f"{x:.6e} ± {y:.6e} ± {math.sqrt(z**2 + d[1]**2):.6e}")
print("\n")
print_acp_result(res, label="A_CP")

eta->gg
-2.058600e-03 ± 6.545800e-03 ± 2.280789e-04
eta->pipipi
1.504400e-03 ± 8.344900e-03 ± 2.681268e-04


== A_CP combination ==
Weights: [0.61904905 0.38095095]
A_CP = -7.012718e-04
  stat         : 5.150351e-03  (rel -734.430%)
  syst (uncorr): 1.671088e-04  (rel -23.829%)
  syst (common): 6.800000e-05  (rel -9.697%)
  syst (total) : 1.804144e-04  (rel -25.727%)
  TOTAL        : 5.153510e-03     (rel -734.881%)

A_CP = (-7.012718e-04 ± 5.150351e-03 (stat) ± 1.671088e-04 (syst-unc) ± 6.800000e-05 (syst-com))
A_CP = (-7.012718e-04 ± 5.150351e-03 (stat) ± 1.804144e-04 (syst-unc-tot)
A_CP = (-7.012718e-04 ± 5.153510e-03)  [total]


In [16]:
# Acp: Ds+ -> eta pi+
a, b, c, d =  -0.13008e-2,  0.48031e-2, math.sqrt(0.019**2 + 0.0086**2 + 0.0)*1e-2, [0.000068, 0.000068] # result1: a ± b (± c ± d)
x, y, z     =  -0.64576e-2, 0.62083e-2, math.sqrt(0.024**2 + 0.0092**2 + 0.0093**2)*1e-2         # result2: x ± y (± z ± d)

res = combine_acp_two_matrix(a, b, c, d, x, y, z)
print(f"eta->gg")
print(f"{a:.6e} ± {b:.6e} ± {math.sqrt(c**2 + d[0]**2):.6e}")
print(f"eta->pipipi")
print(f"{x:.6e} ± {y:.6e} ± {math.sqrt(z**2 + d[1]**2):.6e}")
print("\n")
print_acp_result(res, label="A_CP")

eta->gg
-1.300800e-03 ± 4.803100e-03 ± 2.193627e-04
eta->pipipi
-6.457600e-03 ± 6.208300e-03 ± 2.816682e-04


== A_CP combination ==
Weights: [0.6255807 0.3744193]
A_CP = -3.231605e-03
  stat         : 3.798910e-03  (rel -117.555%)
  syst (uncorr): 1.658198e-04  (rel -5.131%)
  syst (common): 6.800000e-05  (rel -2.104%)
  syst (total) : 1.792211e-04  (rel -5.546%)
  TOTAL        : 3.803135e-03     (rel -117.686%)

A_CP = (-3.231605e-03 ± 3.798910e-03 (stat) ± 1.658198e-04 (syst-unc) ± 6.800000e-05 (syst-com))
A_CP = (-3.231605e-03 ± 3.798910e-03 (stat) ± 1.792211e-04 (syst-unc-tot)
A_CP = (-3.231605e-03 ± 3.803135e-03)  [total]


In [17]:
# Acp: D+ -> eta K+
a, b, c, d =  -8.05044e-2, 8.99226e-2, math.sqrt(0.409**2 + 0.0203**2 + 0.0893**2)*1e-2, [0.000067, 0.000067] # result1: a ± b (± c ± d)
x, y, z     =  5.44115e-2, 9.43647e-2, math.sqrt(0.230**2 + 0.0125**2 + 0.1845**2)*1e-2         # result2: x ± y (± z ± d)

res = combine_acp_two_matrix(a, b, c, d, x, y, z)
print(f"eta->gg")
print(f"{a:.6e} ± {b:.6e} ± {math.sqrt(c**2 + d[0]**2):.6e}")
print(f"eta->pipipi")
print(f"{x:.6e} ± {y:.6e} ± {math.sqrt(z**2 + d[1]**2):.6e}")
print("\n")
print_acp_result(res, label="A_CP")

eta->gg
-8.050440e-02 ± 8.992260e-02 ± 4.191807e-03
eta->pipipi
5.441150e-02 ± 9.436470e-02 ± 2.951972e-03


== A_CP combination ==
Weights: [0.52379277 0.47620723]
A_CP = -1.625647e-02
  stat         : 6.509865e-02  (rel -400.448%)
  syst (uncorr): 2.606667e-03  (rel -16.035%)
  syst (common): 6.700000e-05  (rel -0.412%)
  syst (total) : 2.607528e-03  (rel -16.040%)
  TOTAL        : 6.515085e-02     (rel -400.769%)

A_CP = (-1.625647e-02 ± 6.509865e-02 (stat) ± 2.606667e-03 (syst-unc) ± 6.700000e-05 (syst-com))
A_CP = (-1.625647e-02 ± 6.509865e-02 (stat) ± 2.607528e-03 (syst-unc-tot)
A_CP = (-1.625647e-02 ± 6.515085e-02)  [total]


In [18]:
# Acp: Ds+ -> eta K+
a, b, c, d =  -0.38781e-2, 2.35072e-2, math.sqrt(0.125**2 + 0.0203**2 + 0.0667**2)*1e-2, [0.000067, 0.000067]   # result1: a ± b (± c ± d)
x, y, z     =  -0.34093e-2, 2.98458e-2, math.sqrt(0.117**2 + 0.0125**2 + 0.0736**2)*1e-2         # result2: x ± y (± z ± d)

res = combine_acp_two_matrix(a, b, c, d, x, y, z)
print(f"eta->gg")
print(f"{a:.6e} ± {b:.6e} ± {math.sqrt(c**2 + d[0]**2):.6e}")
print(f"eta->pipipi")
print(f"{x:.6e} ± {y:.6e} ± {math.sqrt(z**2 + d[1]**2):.6e}")
print("\n")
print_acp_result(res, label="A_CP")

eta->gg
-3.878100e-03 ± 2.350720e-02 ± 1.432860e-03
eta->pipipi
-3.409300e-03 ± 2.984580e-02 ± 1.389500e-03


== A_CP combination ==
Weights: [0.61678738 0.38321262]
A_CP = -3.698450e-03
  stat         : 1.846702e-02  (rel -499.318%)
  syst (uncorr): 1.030636e-03  (rel -27.867%)
  syst (common): 6.700000e-05  (rel -1.812%)
  syst (total) : 1.032811e-03  (rel -27.926%)
  TOTAL        : 1.849588e-02     (rel -500.098%)

A_CP = (-3.698450e-03 ± 1.846702e-02 (stat) ± 1.030636e-03 (syst-unc) ± 6.700000e-05 (syst-com))
A_CP = (-3.698450e-03 ± 1.846702e-02 (stat) ± 1.032811e-03 (syst-unc-tot)
A_CP = (-3.698450e-03 ± 1.849588e-02)  [total]


In [3]:
import math
run1 = 1/math.sqrt(428)
run2 = 1/math.sqrt(575.47)

In [7]:
(run1-run2)/run1

0.13759644042921565